In [0]:
import json
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOG = "bucharest_mobility"
BRONZE = "bronze"
SILVER = "silver"

SOURCE_TABLE = f"{CATALOG}.{BRONZE}.weather_snapshots"
TARGET_TABLE = f"{CATALOG}.{SILVER}.weather_hourly"

In [0]:
weather_schema = T.StructType([
    T.StructField("weather_timestamp", T.TimestampType(), False),
    T.StructField("location", T.StringType(), False),
    T.StructField("latitude", T.DoubleType(), False),
    T.StructField("longitude", T.DoubleType(), False),
    T.StructField("temperature_2m", T.DoubleType(), True),
    T.StructField("precipitation", T.DoubleType(), True),
    T.StructField("rain", T.DoubleType(), True),
    T.StructField("snowfall", T.DoubleType(), True),
    T.StructField("weather_code", T.IntegerType(), True),
    T.StructField("wind_speed_10m", T.DoubleType(), True),
    T.StructField("source_snapshot_hash", T.StringType(), False),
    T.StructField("source_ingested_at", T.TimestampType(), False),
])

In [0]:
weather_rows = []

bronze_snapshots = (
    spark.table(SOURCE_TABLE)
    .select(
        "location",
        "latitude",
        "longitude",
        "ingested_at",
        "payload_hash",
        "payload",
    )
    .collect()
)

for snapshot in bronze_snapshots:
    data = json.loads(snapshot["payload"])
    hourly = data["hourly"]

    for i, timestamp in enumerate(hourly["time"]):
        weather_rows.append({
            "weather_timestamp": datetime.fromisoformat(timestamp),
            "location": snapshot["location"],
            "latitude": snapshot["latitude"],
            "longitude": snapshot["longitude"],
            "temperature_2m": hourly["temperature_2m"][i],
            "precipitation": hourly["precipitation"][i],
            "rain": hourly["rain"][i],
            "snowfall": hourly["snowfall"][i],
            "weather_code": hourly["weather_code"][i],
            "wind_speed_10m": hourly["wind_speed_10m"][i],
            "source_snapshot_hash": snapshot["payload_hash"],
            "source_ingested_at": snapshot["ingested_at"],
        })

print(f"Weather hourly rows parsed: {len(weather_rows):,}")

In [0]:
weather_df = spark.createDataFrame(
    weather_rows,
    schema=weather_schema,
)

display(
    weather_df.orderBy("weather_timestamp").limit(10)
)

In [0]:
if spark.catalog.tableExists(TARGET_TABLE):
    existing_hashes = (
        spark.table(TARGET_TABLE)
        .select("source_snapshot_hash")
        .distinct()
    )

    new_weather_df = weather_df.join(
        existing_hashes,
        on="source_snapshot_hash",
        how="left_anti",
    )
else:
    new_weather_df = weather_df

new_rows = new_weather_df.count()

if new_rows == 0:
    print("SKIP weather_hourly: no new snapshots")
else:
    (
        new_weather_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"LOADED weather_hourly: {new_rows:,} rows")

In [0]:
weather_silver = spark.table(TARGET_TABLE)

duplicates = (
    weather_silver
    .groupBy(
        "source_snapshot_hash",
        "location",
        "weather_timestamp",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicates == 0, (
    f"weather_hourly: {duplicates} duplicate grains"
)

print("✓ weather_hourly: grain unique")

In [0]:
assert (
    weather_silver
    .filter(
        (F.col("latitude") < -90)
        | (F.col("latitude") > 90)
        | (F.col("longitude") < -180)
        | (F.col("longitude") > 180)
    )
    .count()
    == 0
)

assert (
    weather_silver
    .filter(F.col("weather_timestamp").isNull())
    .count()
    == 0
)

print("✓ timestamps valid")
print("✓ coordinates valid")